In [1]:
import numpy as np
import cv2 
import glob

# Defining the dimensions of checkerboard
CHECKERBOARD = (9,12)
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)
 
# Creating vector to store vectors of 3D points for each checkerboard image
objpoints = []
# Creating vector to store vectors of 2D points for each checkerboard image
imgpoints = [] 
 
 
# Defining the world coordinates for 3D points
objp = np.zeros((1, CHECKERBOARD[0] * CHECKERBOARD[1], 3), np.float32)
objp[0,:,:2] = np.mgrid[0:CHECKERBOARD[0], 0:CHECKERBOARD[1]].T.reshape(-1, 2)
prev_img_shape = None
 
# Extracting path of individual image stored in a given directory
images = glob.glob('./image/*.jpg')
for fname in images:
    img = cv2.imread(fname)
    gray = cv2.cvtColor(img,cv2.COLOR_BGR2GRAY)
    # Find the chess board corners
    # If desired number of corners are found in the image then ret = true
    ret, corners = cv2.findChessboardCorners(gray, CHECKERBOARD, cv2.CALIB_CB_ADAPTIVE_THRESH + cv2.CALIB_CB_FAST_CHECK + cv2.CALIB_CB_NORMALIZE_IMAGE)
     
    """
    If desired number of corner are detected,
    we refine the pixel coordinates and display 
    them on the images of checker board
    """
    if ret == True:
        objpoints.append(objp)
        # refining pixel coordinates for given 2d points.
        corners2 = cv2.cornerSubPix(gray, corners, (11,11),(-1,-1), criteria)
         
        imgpoints.append(corners2)
 
        # Draw and display the corners
        img = cv2.drawChessboardCorners(img, CHECKERBOARD, corners2, ret)
        cv2.imshow('img',img)
        cv2.waitKey(1000)
     
cv2.destroyAllWindows()
 
h,w = img.shape[:2]
 
"""
Performing camera calibration by 
passing the value of known 3D points (objpoints)
and corresponding pixel coordinates of the 
detected corners (imgpoints)
"""
ret, mtx, dist, rvecs, tvecs = cv2.calibrateCamera(objpoints, imgpoints, gray.shape[::-1], None, None)


print("ret : \n")
print(ret)
print("Camera matrix : \n")
print(mtx)
print("dist : \n")
print(dist)
print("rvecs : \n")
print(rvecs)
print("tvecs : \n")
print(tvecs)

ret : 

1.1356197113800057
Camera matrix : 

[[871.14455645   0.         677.41224863]
 [  0.         754.80942595 468.71784497]
 [  0.           0.           1.        ]]
dist : 

[[ 0.1236253  -0.08203174  0.04473649  0.03499051  0.0280648 ]]
rvecs : 

(array([[ 0.06426787],
       [ 0.01849825],
       [-1.6965826 ]]), array([[-0.16406602],
       [-0.22802549],
       [-1.82466863]]), array([[ 0.25629521],
       [-0.51230416],
       [-1.51058765]]), array([[-0.21907684],
       [ 0.21331084],
       [-1.68848503]]))
tvecs : 

(array([[-14.9479731 ],
       [ -4.29461801],
       [ 31.92689287]]), array([[-10.62092773],
       [  1.53552311],
       [ 23.36563511]]), array([[-8.23391985],
       [-2.14387664],
       [16.18009602]]), array([[-11.23430852],
       [  2.0009535 ],
       [ 24.06659853]]))


In [6]:
img= cv2.imread('./image/WIN_20240326_15_01_14_Pro.jpg')
h, w= img.shape[:2]
newcameraMatrix, roi = cv2.getOptimalNewCameraMatrix(mtx, dist, (w,h), 1, (w,h))

dst = cv2.undistort(img, mtx, dist, None, newcameraMatrix)
x, y, w, h = roi
dst = dst[y:y+h, x:x+w]
cv2.imwrite('cr.jpg', dst)


# mean_err = 0
# for i in range(len(objpoints)):
#     imgPoints2= cv2.projectPoints(objpoints[i], rvecs[i], tvecs[i], mtx, dist)
#     err = cv2.norm(imgpoints[i], imgPoints2[i], cv2.NORM_L2)/len(imgPoints2)
#     mean_err += err

# print("\ntotal err: {}".format(mean_err/len(objpoints)))
# print("\n\n\n")

True